# Decision Trees & Random Forests — Simple Hands-On Notebook

This notebook is a shorter, simpler version, built to be followed step by step with almost no extra explanation needed.

**What we'll do, in order:**
1. Build a small, easy-to-understand dataset
2. Train ONE decision tree with no limits, and watch it "overfit"
3. Look at the actual tree it built
4. Limit how deep the tree is allowed to grow, and see accuracy improve
5. Grow a Random Forest (many trees voting together) and compare it
6. See which feature the model actually relied on most

Run each cell top to bottom using **Shift + Enter**. Every line of code has a comment above or beside it explaining exactly what it does.

## Step 1 — Import the Tools We Need

In [ ]:
# pandas lets us store data in a neat table, like a spreadsheet
import pandas as pd

# numpy helps us generate random numbers for our practice dataset
import numpy as np

# matplotlib is what we use to draw charts and pictures
import matplotlib.pyplot as plt

# train_test_split cuts our data into a "practice" part and a "test" part
from sklearn.model_selection import train_test_split

# DecisionTreeClassifier builds a single decision tree
# plot_tree lets us draw that tree as a picture
from sklearn.tree import DecisionTreeClassifier, plot_tree

# RandomForestClassifier builds many decision trees that vote together
from sklearn.ensemble import RandomForestClassifier

# accuracy_score tells us what percentage of predictions were correct
from sklearn.metrics import accuracy_score

# lock in the "randomness" so everyone gets the exact same numbers every time
np.random.seed(42)

# a simple message to confirm everything loaded properly
print("Ready to go!")

## Step 2 — Build a Simple Practice Dataset

We'll pretend we have data on 300 students. Each student has two numbers about them:

- `hours_studied` — how many hours they studied
- `attendance_pct` — what percent of classes they attended

And one final result: did they `passed` (1) or not (0)?

The rule we use to decide "pass or fail" is simple: study more and attend more, and you're more likely to pass — but we add a bit of randomness too, because real life is never perfectly predictable.

In [ ]:
# how many students we want in our practice dataset
n_students = 300

# generate 300 random "hours studied" values, centered around 5 hours
# np.random.normal picks numbers that cluster around the middle, like most real data does
# np.clip stops any value from going below 0 or above 10 (you can't study negative hours)
hours_studied = np.clip(np.random.normal(5, 2.3, n_students), 0, 10).round(1)

# generate 300 random "attendance percent" values, centered around 75%
attendance_pct = np.clip(np.random.normal(75, 18, n_students), 0, 100).round(0)

# turn both numbers into a 0-to-1 scale so we can combine them fairly
# (10 hours studied becomes 1.0, 100% attendance becomes 1.0)
hours_score = hours_studied / 10
attendance_score = attendance_pct / 100

# combine both scores equally -- studying and attending matter the same amount here
combined_score = 0.5 * hours_score + 0.5 * attendance_score

# add a small random "wobble" to each student's score, so the pattern isn't perfectly clean
random_wobble = np.random.normal(0, 0.15, n_students)
final_score = combined_score + random_wobble

# a student "passes" (1) if their final score is above 0.5, otherwise they "fail" (0)
passed = (final_score > 0.5).astype(int)

# put everything together into one table (a pandas DataFrame)
students = pd.DataFrame({
    "hours_studied": hours_studied,
    "attendance_pct": attendance_pct,
    "passed": passed
})

# print what percentage of our students passed, just to sanity-check the data
print("Percent of students who passed:", students["passed"].mean().round(2))

# show the first 10 rows of the table so we can see what it looks like
students.head(10)

## Step 3 — Split the Data into Practice and Test Sets

In [ ]:
# X holds the two features the model is allowed to look at
X = students[["hours_studied", "attendance_pct"]]

# y holds the correct answer (passed or not) that we're trying to predict
y = students["passed"]

# split into 70% training data (to learn from) and 30% test data (to check on afterward)
# random_state=42 makes sure this split comes out the same way every time we run it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# print how many students ended up in each part
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## Step 4 — Train ONE Tree With No Limits, and Watch It Overfit

We'll let this tree ask as many questions as it wants, with no restrictions. Then we check its accuracy on the data it TRAINED on versus data it has NEVER SEEN before (the test set).

In [ ]:
# create a decision tree with no depth limit -- it can keep splitting forever
unlimited_tree = DecisionTreeClassifier(max_depth=None, random_state=42)

# train the tree using only the training data
unlimited_tree.fit(X_train, y_train)

# ask the tree to predict on the SAME data it just trained on
train_predictions = unlimited_tree.predict(X_train)

# ask the tree to predict on the test data it has never seen before
test_predictions = unlimited_tree.predict(X_test)

# calculate how many training predictions were correct
train_accuracy = accuracy_score(y_train, train_predictions)

# calculate how many test predictions were correct
test_accuracy = accuracy_score(y_test, test_predictions)

# print both accuracies so we can compare them
print(f"Accuracy on training data: {train_accuracy:.0%}")
print(f"Accuracy on test data:     {test_accuracy:.0%}")

# print how many levels of questions the tree ended up asking
print(f"The tree grew {unlimited_tree.get_depth()} levels deep to do this!")

**What this shows:** the tree scores very high on the training data (the data it memorized) but noticeably lower on the test data (new students it has never seen). That gap is called **overfitting** — the tree learned the training examples too specifically instead of learning a general rule.

## Step 5 — Look at the Actual Tree It Built

In [ ]:
# make the picture big enough to read
plt.figure(figsize=(18, 8))

# draw the tree as a flowchart picture
plot_tree(
    unlimited_tree,                       # the tree we trained above
    feature_names=X.columns,              # use real names instead of X[0], X[1]
    class_names=["Fail", "Pass"],         # use real words instead of 0 and 1
    filled=True,                          # color each box based on its prediction
    max_depth=3,                          # only draw the first 3 levels so it's readable
    fontsize=9                            # keep the text a reasonable size
)

# add a title explaining what we're looking at
plt.title("The Unlimited Tree (only the first 3 levels are shown -- it goes much deeper)")

# actually display the picture
plt.show()

## Step 6 — Limit the Tree's Depth, and Compare

Now let's train a few trees, each allowed to ask fewer questions (a smaller `max_depth`), and compare their test accuracy to the unlimited tree above.

In [ ]:
# the different depth limits we want to try
depths_to_try = [2, 4, 8]

# go through each depth limit one at a time
for depth in depths_to_try:

    # create a new tree limited to this depth
    limited_tree = DecisionTreeClassifier(max_depth=depth, random_state=42)

    # train it on the training data
    limited_tree.fit(X_train, y_train)

    # check its accuracy on the training data
    this_train_accuracy = accuracy_score(y_train, limited_tree.predict(X_train))

    # check its accuracy on the test data
    this_test_accuracy = accuracy_score(y_test, limited_tree.predict(X_test))

    # print a clean summary line for this depth
    print(f"max_depth={depth}:  train={this_train_accuracy:.0%}   test={this_test_accuracy:.0%}")

# for comparison, print the unlimited tree's numbers again right below
print(f"max_depth=None (unlimited):  train={train_accuracy:.0%}   test={test_accuracy:.0%}")

**What to notice:** the limited trees have LOWER training accuracy than the unlimited tree (they memorize less), but their test accuracy is usually just as good, or better. `max_depth` is a setting you choose yourself — this is exactly how you'd pick a good value for it on a real project: try a few, and keep the one that does best on the test data.

## Step 7 — Grow a Random Forest

A single tree can be unstable. A Random Forest fixes this by training many different trees and letting them vote on the final answer.

In [ ]:
# n_estimators = how many different trees to grow in the forest
# each tree secretly gets a slightly different, randomly chosen slice of the training data
forest = RandomForestClassifier(n_estimators=200, random_state=42)

# train the whole forest on the training data
forest.fit(X_train, y_train)

# check the forest's accuracy on the training data
forest_train_accuracy = accuracy_score(y_train, forest.predict(X_train))

# check the forest's accuracy on the test data
forest_test_accuracy = accuracy_score(y_test, forest.predict(X_test))

# print the forest's results
print(f"Random Forest:  train={forest_train_accuracy:.0%}   test={forest_test_accuracy:.0%}")

# print the single unlimited tree's results again, right below, for an easy comparison
print(f"Single Tree (unlimited):  train={train_accuracy:.0%}   test={test_accuracy:.0%}")

**What to notice:** the forest's test accuracy is usually better than (or very close to) the single unlimited tree's — even though we didn't bother limiting each individual tree's depth. Voting across many different trees cancels out each tree's individual mistakes.

## Step 8 — Which Feature Did the Model Rely On Most?

In [ ]:
# feature_importances_ gives one score per feature, showing how much the forest relied on it
# the two scores always add up to 1.0 (100%)
importance_scores = pd.Series(forest.feature_importances_, index=X.columns)

# sort so the most important feature is listed first
importance_scores = importance_scores.sort_values(ascending=False)

# print the exact numbers
print(importance_scores.round(2))

In [ ]:
# make a simple horizontal bar chart of the importance scores
plt.figure(figsize=(7, 3))

# sort ascending here just so the biggest bar visually ends up on top
importance_scores.sort_values().plot(kind="barh", color="#306998")

# label the x-axis
plt.xlabel("Importance")

# give the chart a title
plt.title("What the Random Forest Relied On Most")

# tidy up the spacing so nothing gets cut off
plt.tight_layout()

# display the chart
plt.show()

Since we built `hours_studied` and `attendance_pct` to matter **equally** when we created this dataset back in Step 2, the two bars should come out fairly close to each other — proof the forest is picking up on the real pattern in the data, not something random.

## Recap

- A **Decision Tree** is a chain of yes/no questions that ends in a prediction.
- An **unlimited tree** can memorize the training data perfectly, but that hurts its accuracy on new data — this is called **overfitting**.
- **`max_depth`** controls how many questions the tree is allowed to ask — limiting it usually helps the tree generalize better.
- A **Random Forest** trains many different trees and lets them vote, which is usually more accurate and more reliable than any single tree.
- **`feature_importances_`** tells you which features the model actually relied on.

That's the whole workflow: build data, train a tree, watch it overfit, fix it, then grow a forest.